## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 84 samples and 13998 columns.
First few rows of the data:


/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_41665/1856502050.py:30: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(preprocessed_csv_path)


,pfs.mos..,recurrence,os.mos..,vitalstatus,sample_title,stic,age,path.stage,surgical.outcome,diagnosis.of.record,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,19.0,1,34.0,1.0,HGSC_40190,Yes,61,III,Optimal,Not available,...,6.040067,5.035451,5.740524,6.091217,4.829373,5.089559,6.619170,6.902299,6.207234,6.313062
1,40.0,1,64.0,0.0,HGSC_40191,No,49,III,Optimal,Not available,...,6.537236,5.038692,5.921121,6.717286,5.075715,5.334657,6.859956,6.920267,6.185481,6.279690
2,42.0,1,72.0,0.0,HGSC_40193,No,53,II,Unknown,Not available,...,7.070010,6.296203,7.057889,6.890579,5.866339,6.279101,7.889931,8.161322,7.570871,7.458223
3,32.0,1,34.0,1.0,HGSC_40197,Yes,53,III,Optimal,Not available,...,6.976777,6.125367,6.876939,7.116488,5.835287,6.258743,7.824677,8.059266,7.367960,7.466249
4,13.0,1,39.0,1.0,HGSC_40199,No,49,III,Suboptimal,Not available,...,6.490908,5.032042,5.987931,6.960127,5.560689,5.339411,6.856786,7.024983,6.217915,6.551331


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
